<a href="https://colab.research.google.com/github/TalCordova/RS_Coller_TAU_26B/blob/master/notebook4_content_based_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content-Based Filtering: Recommending by Item Features

So far we built **collaborative filtering** recommenders (item-item and user-user). They all read from the same place: the **utility matrix** of user–item interactions. Two movies are "similar" if *users rated them similarly*.

Content-based filtering keeps the **exact same machinery** — build a similarity between items, take the nearest ones, recommend *"Because you watched X…"* — but swaps out the **signal**:

| | Item-Item CF | Content-Based |
|---|---|---|
| Two items are similar if… | users rated them similarly | their **features** match |
| Signal source | the utility matrix (interactions) | item metadata (here: **genres**) |
| Needs interactions on the item? | **yes** | **no** |

That last row is the whole point. A brand-new movie with zero ratings is invisible to item-item CF — but it still has genres, so content-based can rank it immediately. (We'll come back to this when we discuss the cold-start problem.)

> **Scope:** this is the same item→item *"Because you watched X"* shape you already know, now driven by content. We keep preprocessing deliberately minimal — the goal is the *idea*, not a production feature pipeline.

---
> **How to use this notebook:** read the markdown, run each cell in order, and complete the 🏋️ exercise.

## Section 0: Setup

In [ ]:
!pip install numpy pandas scikit-learn --quiet

In [2]:
import ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors

pd.set_option('display.max_colwidth', 60)
print("✅ Packages loaded")

✅ Packages loaded


## Section 1: Load the Data

We use the **`movies_metadata.csv`** file from Kaggle's *The Movies Dataset* (metadata for ~45,000 movies, sourced from TMDB). Unlike the MovieLens ratings we used for collaborative filtering, this file is pure **item metadata**: no users, no interactions at all. That's exactly what we want: a different signal.

The file has been shared with you via Google Drive. Update the path below to match where you saved it in **your own Drive**.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Update this to wherever you placed the file in your Drive
METADATA_PATH = "/content/drive/YOUR_PATH_HERE/movies_metadata.csv"

# low_memory=False avoids the mixed-dtype warning this file is known for
movies = pd.read_csv(METADATA_PATH, low_memory=False)
print(f"Loaded {len(movies):,} movies")
movies[['title', 'genres']].head()

Look at the `genres` column above. It is **not** a clean list — each cell is a *stringified* list of dictionaries, e.g.

```
[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]
```

We only care about the genre **names**, so our one preprocessing job is to pull those out.

In [3]:
# If you run local
movies = pd.read_csv('movies_metadata.csv', low_memory=False)
movies.head()

,Unnamed: 0,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,...,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,genres_tmp
0,0,FALSE,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_pa...",30000000,"['Animation', 'Comedy', 'Family']",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,...,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"['Animation', 'Comedy', 'Family']"
1,1,FALSE,NaN,65000000,"['Adventure', 'Fantasy', 'Family']",NaN,8844,tt0113497,en,Jumanji,...,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': '...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"['Adventure', 'Fantasy', 'Family']"
2,2,FALSE,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'pos...",0,"['Romance', 'Comedy']",NaN,15602,tt0113228,en,Grumpier Old Men,...,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for Love.,Grumpier Old Men,False,6.5,92.0,"['Romance', 'Comedy']"
3,3,FALSE,NaN,16000000,"['Comedy', 'Drama', 'Romance']",NaN,31357,tt0114885,en,Waiting to Exhale,...,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself... and ne...,Waiting to Exhale,False,6.1,34.0,"['Comedy', 'Drama', 'Romance']"
4,4,FALSE,"{'id': 96871, 'name': 'Father of the Bride Collection', ...",0,['Comedy'],NaN,11862,tt0113041,en,Father of the Bride Part II,...,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's In For The...,Father of the Bride Part II,False,5.7,173.0,['Comedy']


## Section 2: Minimal Preprocessing — Genres → Multi-Hot

Two small steps:
1. **Parse** each `genres` string into a plain list of genre names (and quietly drop the handful of corrupt rows this file contains).
2. **Multi-hot encode**: turn the genre lists into a 0/1 matrix — one column per genre, a 1 where the movie has that genre.

In [4]:
def parse_genres(value):
    """Turn a stringified list-of-dicts into a list of genre names.
    Returns [] for the corrupt / malformed rows in this file."""
    try:
        parsed = ast.literal_eval(value)
        return [g['name'] if isinstance(g, dict) else g for g in parsed]
    except (ValueError, SyntaxError):
        return []

movies['genre_list'] = movies['genres'].apply(parse_genres)

# Keep only movies that actually have genres, and one row per title
before = len(movies)
movies = movies[movies['genre_list'].map(len) > 0]
movies = movies.drop_duplicates(subset='title').reset_index(drop=True)
print(f"Kept {len(movies):,} movies (dropped {before - len(movies):,} with no usable genres / duplicates)")
movies[['title', 'genre_list']].head()

Kept 40,034 movies (dropped 5,432 with no usable genres / duplicates)


,title,genre_list
0,Toy Story,"[Animation, Comedy, Family]"
1,Jumanji,"[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Romance, Comedy]"
3,Waiting to Exhale,"[Comedy, Drama, Romance]"
4,Father of the Bride Part II,[Comedy]


In [5]:
# Multi-hot encode: one binary column per genre
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genre_list'])

print(f"Genre matrix shape: {genre_matrix.shape}  (movies × genres)")
print(f"Genres ({len(mlb.classes_)}): {', '.join(mlb.classes_)}")

# Peek: Toy Story as a row of 0s and 1s
example = movies.index[movies['title'] == 'Toy Story'][0]
pd.Series(genre_matrix[example], index=mlb.classes_, name='Toy Story').to_frame().T

Genre matrix shape: (40034, 20)  (movies × genres)
Genres (20): Action, Adventure, Animation, Comedy, Crime, Documentary, Drama, Family, Fantasy, Foreign, History, Horror, Music, Mystery, Romance, Science Fiction, TV Movie, Thriller, War, Western


,Action,Adventure,Animation,Comedy,Crime,Documentary,Drama,Family,Fantasy,Foreign,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
Toy Story,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


## Section 3: Build the Similarity

Each movie is now a vector in *genre space*. We measure how close two movies are with **cosine similarity** — the same metric we used for item-item CF. The only thing that changed is what the vectors are built from: **genres instead of ratings**.

We fit `NearestNeighbors` so we can look up the closest movies on demand, without ever materializing a giant 45,000 × 45,000 matrix.

In [6]:
nn = NearestNeighbors(metric='cosine', algorithm='brute')
nn.fit(genre_matrix)
print("✅ Similarity index ready")

✅ Similarity index ready


## Section 4: *"Because you watched X…"*

The recommendation step is identical in spirit to item-item CF: take a movie, find its nearest neighbors, return them.

In [10]:
title_to_index = pd.Series(movies.index, index=movies['title'])

def content_recommendations(title, k=10):
    """Return the k movies most similar to `title` by genre content."""
    if title not in title_to_index:
        raise KeyError(f"'{title}' not found. Check the exact title.")
    idx = title_to_index[title]

    distances, indices = nn.kneighbors([genre_matrix[idx]], n_neighbors=k + 1)
    similarities = 1 - distances.flatten()

    results = []
    for sim, i in zip(similarities, indices.flatten()):
        if i == idx:           # skip the movie itself
            continue
        results.append({'title': movies.loc[i, 'title'],
                        'genres': ', '.join(movies.loc[i, 'genre_list']),
                        'similarity': round(sim, 3)})
    return pd.DataFrame(results[:k])

In [11]:
print("Because you watched Inception:")
content_recommendations('Inception', k=10)

Because you watched Inception:


,title,genres,similarity
0,Paycheck,"Action, Adventure, Mystery, Science Fiction, Thriller",1.000
1,Sky Captain and the World of Tomorrow,"Mystery, Action, Thriller, Science Fiction, Adventure",1.000
2,Knowing,"Action, Adventure, Drama, Mystery, Science Fiction, Thri...",0.913
3,Congo,"Action, Adventure, Drama, Mystery, Science Fiction, Thri...",0.913
4,Zenith,"Action, Adventure, Drama, Mystery, Science Fiction, Thri...",0.913
5,Star Trek III: The Search for Spock,"Science Fiction, Action, Adventure, Thriller",0.894
6,The Abyss,"Adventure, Action, Thriller, Science Fiction",0.894
7,Bear Island,"Mystery, Adventure, Thriller, Action",0.894
8,Starship Troopers,"Adventure, Action, Thriller, Science Fiction",0.894
9,ISRA 88,"Thriller, Adventure, Science Fiction, Mystery",0.894


Look at the **scores**, not just the titles. Movies that share *all five* of Inception's genres come back at **1.0**, while ones that match *most but not all* land a notch lower (**0.913**, **0.894**). So the recommender genuinely **ranks by how much genre overlap there is** — it isn't just dumping a random pile of same-genre movies.

And notice it needed **no ratings at all** — only genres. We could recommend a movie that *nobody has ever watched*, as long as we know its genres. That is something collaborative filtering simply cannot do.

> Try a few **other** titles, though, and you'll often get a wall of identical **1.0**s. That's not a bug — it's exactly what the exercise below asks you to explain.

### 🏋️ Exercise: Read the Output Critically

Run `content_recommendations` on a few movies of your choice and look hard at the results.

1. Many movies will come back with **similarity = 1.0**. Why? What does a perfect cosine score mean when the vectors are just multi-hot genres?
2. This is the main *weakness* of pure genre content: it cannot tell apart two movies that share the same genre set. List two things you'd add to the item vector to break these ties.

Write your answers in the cell below.

In [9]:
# Try a few titles — display both so we can eyeball the genres side by side:
print("Because you watched Jumanji:")
display(content_recommendations('Jumanji', k=10))

print("\nBecause you watched GoldenEye:")
display(content_recommendations('GoldenEye', k=10))

Because you watched Jumanji:


,title,genres,similarity
0,The Indian in the Cupboard,"Adventure, Family, Fantasy",1.0
1,Jason and the Argonauts,"Adventure, Family, Fantasy",1.0
2,Labyrinth,"Adventure, Family, Fantasy",1.0
3,The Chronicles of Narnia: Prince Caspian,"Adventure, Family, Fantasy",1.0
4,Cry Wilderness,"Fantasy, Adventure, Family",1.0
5,Percy Jackson: Sea of Monsters,"Adventure, Family, Fantasy",1.0
6,The Spiderwick Chronicles,"Adventure, Family, Fantasy",1.0
7,Harry Potter and the Prisoner of Azkaban,"Adventure, Fantasy, Family",1.0
8,The Boy and the Pirates,"Fantasy, Family, Adventure",1.0
9,Journey to the Beginning of Time,"Adventure, Fantasy, Family",1.0



Because you watched GoldenEye:


,title,genres,similarity
0,Angel of Destruction,"Adventure, Action, Thriller",1.0
1,Blood Out,"Action, Adventure, Thriller",1.0
2,The Hunt for Eagle One: Crash Point,"Action, Adventure, Thriller",1.0
3,The Art of War III: Retribution,"Adventure, Action, Thriller",1.0
4,A View to a Kill,"Adventure, Action, Thriller",1.0
5,Titanic 2,"Action, Adventure, Thriller",1.0
6,Firestorm,"Action, Adventure, Thriller",1.0
7,Dr. No,"Adventure, Action, Thriller",1.0
8,From Russia with Love,"Action, Thriller, Adventure",1.0
9,Goldfinger,"Adventure, Action, Thriller",1.0


### 🔑 Solution

<details>
<summary>🔑 Reveal answers</summary>

**1. Why so many `similarity = 1.0`?**

Cosine similarity only looks at the *direction* of the vectors, and a multi-hot genre vector encodes nothing but *which genres are present*. So two movies that share the **exact same set of genres** have the **exact same vector** — and the cosine between identical vectors is **1.0**.

This happens constantly here because the signal is so coarse: there are only ~20 genres and most movies carry just 1–3 of them. Every `Comedy, Romance` movie looks *identical* to every other `Comedy, Romance` movie. A perfect score doesn't mean "these are the same film" — it means "genres alone can't tell these apart."

**2. Two things to add to the item vector to break the ties** *(staying with simple numeric metadata — no text)*

- **Release year** (from `release_date`) — separates a 1995 comedy from a 2015 comedy.
- **`vote_count`** — a popularity signal; distinguishes a blockbuster from an obscure title in the same genre.
- **`vote_average`** — the average rating; separates a well-loved film from a poorly-rated one.

(Any two of these are enough.) One caveat: these are continuous and live on much larger scales than the 0/1 genre columns, so **normalize/standardize them first** before concatenating — otherwise a raw `vote_count` in the thousands would swamp the genre bits and dominate the cosine.

</details>

## Section 5: Where This Fits

**Content-based is another signal source, not a better algorithm.** It sits alongside item-item and user-user CF: same retrieval machinery, different input.

- **vs. CF:** CF needs interactions; content does not. CF can capture taste patterns genres miss ("people who like *this* obscure film also like…"); content can recommend brand-new or never-watched items. They are complementary — real systems blend them.
- **Cold start (next topic):** because content similarity comes from features, it handles the **item** cold-start case directly — a new movie is rankable the moment we know its genres. Note this fixes the *item* side only: a brand-new **user** with no history still has no profile to match against.

### Going further (not needed for this tutorial)
Genres are a coarse signal — hence all the 1.0 ties in the exercise. The natural upgrade is richer item features: the `overview` text, cast/crew, keywords. You could encode the `overview` with TF-IDF or sentence embeddings (e.g. a BERT-family model) to get a much finer-grained similarity. Same machinery — just a richer item vector.